# 03 — Train YOLOv8 Baseline
Train YOLOv8n on NEU-DET. On a GPU this takes ~30–60 min for 100 epochs; on CPU it is much slower (reduce epochs to test the pipeline).

## 1. Check environment

In [1]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA RTX 2000 Ada Generation


## 2. Locate dataset config

In [2]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_CFG = ROOT / 'data' / 'neu-det-yolo' / 'data.yaml'
assert DATA_CFG.exists(), 'Run 01_data_preparation.ipynb first!'
print('Data config:', DATA_CFG)
print(DATA_CFG.read_text())

Data config: c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml
path: c:/Users/student/Desktop/SteelDefectDetection/data/neu-det-yolo
train: images/train
val: images/val
test: images/test

nc: 6
names:
  0: crazing
  1: inclusion
  2: patches
  3: pitted_surface
  4: rolled-in_scale
  5: scratches



## 3. Load a pretrained YOLOv8n model
We start from COCO-pretrained `yolov8n.pt` (transfer learning) — far better than training from scratch on only 1,800 images.

In [3]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')   # downloads weights on first run
print('Loaded YOLOv8n')

Loaded YOLOv8n


## 4. Train
Key settings (matched to the paper, tuned for a ~16 GB GPU):
- `epochs=150` — lower to 20–30 for a quick test
- `imgsz=640` — **paper resolution.** NEU-DET is 200×200; training at 640 (instead of the old 224) restores the fine detail the low-contrast classes (*crazing*, *rolled-in_scale*) need. The previous 224 run reached only **~0.64** mAP@0.5 and early-stopped at epoch 51; 640 should reach roughly **~0.74–0.77** (the paper's baseline is 0.774).
- `batch=16` — 640px is much heavier than 224, so 64 would OOM. 16 fits a 16 GB GPU; drop to 8/4 if you still OOM, or try 32 if you have headroom.
- `cache=True` — dataset is ~40 MB, so cache it in RAM for faster epochs
- `workers=0` — Windows-safe: avoids the `close_mosaic` data-loader deadlock at epoch 90
- results saved to `results/baseline_640/` — your old 224 run in `results/baseline/` is left untouched

In [4]:
# imgsz=640 (paper setting). batch=16 fits 16 GB VRAM at 640px; drop to 8/4 if you OOM.
BATCH = 16 if DEVICE == 0 else 4   # smaller batch on CPU just to exercise the pipeline

results = model.train(
    data=str(DATA_CFG),
    epochs=150,
    imgsz=640,          # paper resolution; 224 only reached ~0.64 mAP (NEU-DET 200x200 upscaled to 640)
    batch=BATCH,
    device=DEVICE,
    cache=True,         # tiny dataset (~40 MB) -> cache in RAM for faster epochs
    workers=0,          # Windows-safe: avoids the close_mosaic data-loader deadlock at epoch 90
    project=str(ROOT / 'results'),
    name='baseline_640',
    exist_ok=True,      # reuse results/baseline_640 on re-run (one clean output dir)
    patience=50,        # early stop if no improvement
    seed=42,
    optimizer='SGD',    # SGD+momentum (lr0=0.01): standard YOLO detection optimizer; 'auto' picks AdamW on small data and tends to undershoot final mAP
    cos_lr=True,        # cosine LR decay -> smoother convergence (helps from-scratch backbones)
    plots=True,
)
print('Training done. Best weights:', ROOT / 'results/baseline_640/weights/best.pt')

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8

## 5. Validation summary (model-selection set, 180 imgs)
This is the `val` split YOLO used *during* training for early-stopping / best-epoch selection.
It is **not** the paper-comparable number — see section 6 for the held-out test set.

In [5]:
# Validation set (180 imgs) — reload best.pt so this runs WITHOUT re-training (fresh kernel OK).
from ultralytics import YOLO
best = ROOT / 'results' / 'baseline_640' / 'weights' / 'best.pt'
metrics = YOLO(str(best)).val(data=str(DATA_CFG), split='val')
print('--- VALIDATION set (180 imgs) ---')
print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'precision    : {metrics.box.mp:.4f}')
print(f'recall       : {metrics.box.mr:.4f}')

Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 294.581.4 MB/s, size: 12.8 KB)
val: Scanning C:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\labels\val.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 6.7it/s 1.8s0.1s
                   all        180        441      0.716      0.655      0.725      0.389
               crazing         30         82      0.579      0.244      0.447      0.158
             inclusion         35         87       0.79      0.782      0.864      0.467
               patches         36        100      0.821       0.84      0.908      0.586
        pitted_surface         30         41      0.898      0.707       0.79      0

## 6. Test-set evaluation — the paper's reported metric
The paper reports results on the held-out **test** split (180 imgs, never seen during training or
model selection). We reload `best.pt` and evaluate with `split='test'`. The baseline target is the
paper's **mAP@0.5 = 0.774**.

In [6]:
# TEST set (180 imgs) — held out; THIS is the number comparable to the paper.
from ultralytics import YOLO
CLASSES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']

best = ROOT / 'results' / 'baseline_640' / 'weights' / 'best.pt'
test_model = YOLO(str(best))                      # reload best checkpoint
tm = test_model.val(data=str(DATA_CFG), split='test')

print('--- TEST set (180 imgs) — paper-comparable ---')
print(f'mAP@0.5      : {tm.box.map50:.4f}   (paper baseline: 0.774)')
print(f'mAP@0.5:0.95 : {tm.box.map:.4f}')
print(f'precision    : {tm.box.mp:.4f}')
print(f'recall       : {tm.box.mr:.4f}')
print('\nPer-class mAP@0.5:')
for i, name in enumerate(CLASSES):
    print(f'  {name:<18}{float(tm.box.ap50[i]):.4f}')

Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 50.518.4 MB/s, size: 12.4 KB)
val: Scanning C:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\labels\test.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 8.2it/s 1.5s0.2s
                   all        180        413      0.743      0.665      0.752      0.393
               crazing         30         80      0.574        0.3       0.44      0.161
             inclusion         36         72      0.827      0.764      0.858      0.476
               patches         37         93      0.869      0.859      0.925      0.603
        pitted_surface         30         46      0.924      0.696      0.862      0

✅ **Training complete.** Next: open `04_evaluate.ipynb` for detailed analysis.